# Nestlé WISER DOM experiment laboratory

This notebook is self-contained: use **Run All** after editing the global configuration
cell directly below the dependency bootstrap. Missing local notebook dependencies are
installed by the kernel; no terminal commands or terminal-set environment variables are
required. It reads the five canonical runtime CSV inputs and, when present, two optional
recommendation-output CSVs. Only aggregate metrics and figures are saved.

Notebook evidence is stored in the stable, human-readable directory
`results/challenge-study/notebook/<profile>/`. Start with `PROFILE = "smoke"`; change
it to `"full"` for final evidence. Verified per-experiment checkpoints let a long run
resume without creating nested problem/run hash directories.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path


def _project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/domopt").is_dir():
            return candidate
    raise RuntimeError(
        "Open this notebook from inside the wiser-dom-optimization repository"
    )


PROJECT_ROOT = _project_root(Path.cwd())
source_directory = str(PROJECT_ROOT / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

required_modules = ("numpy", "pandas", "scipy", "yaml", "matplotlib", "nbformat")
missing_modules = [
    module for module in required_modules if importlib.util.find_spec(module) is None
]
if missing_modules:
    print(f"Installing missing notebook dependencies: {', '.join(missing_modules)}")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-e",
            f"{PROJECT_ROOT}[notebook]",
        ],
        check=True,
    )
print(f"Notebook environment ready: {PROJECT_ROOT}")


In [ ]:
from dataclasses import asdict, replace
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from domopt.checkpoints import (
    StaleCheckpointError,
    challenge_results_root,
    checkpoint_identity,
    checkpoint_run_directory,
    load_checkpoint,
    write_checkpoint,
)
from domopt.classical import (
    ClassicalSolverError,
    available_milp_backends,
    solve_classical,
)
from domopt.experiments import (
    experiment_profile,
    ibm_hardware_study_logical_qubits,
    make_ibm_hardware_study_problem,
    rank_ibm_hardware_strategies,
    run_challenge_experiments,
    run_ibm_hardware_study,
    write_experiment_results,
)
from domopt.hardware import (
    benchmark_qubo_batch_scoring,
    discover_ibm_backends,
    hardware_capabilities,
)
from domopt.metrics import compute_metrics
from domopt.poc import (
    POC_REFERENCE_FILENAMES,
    PocConfig,
    audit_poc_bundle,
    audit_poc_outputs,
    load_poc_problem,
    prune_pareto_candidates,
    select_shortage_subset,
)
from domopt.visualization import (
    plot_challenge_results,
    plot_hardware_benchmark,
    plot_ibm_backend_snapshot,
    plot_ibm_hardware_study,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "src/domopt").is_dir()
        ):
            return candidate
    raise RuntimeError(
        "Run this notebook from inside the wiser-dom-optimization repository"
    )


# -----------------------------------------------------------------------------
# GLOBAL CONFIGURATION — edit only this block, then use Run All.
# -----------------------------------------------------------------------------
PROJECT_ROOT = find_project_root(Path.cwd())
BUNDLE_DIR = (PROJECT_ROOT / "data/raw/nestle_challenge").resolve()
PROFILE = "smoke"  # "smoke" for a quick check; "full" for final evidence
FORCE_RERUN = False
POC_SETTINGS = PocConfig(
    protection_days=5,
    min_divert_improvement_fraction=0.05,
    min_divert_improvement_cases=100,
    candidate_dc_scope="network_intersection",
    pareto_prune=False,
)
# Override any profile field without editing package code. Examples:
# PROFILE_OVERRIDES = {"sizes": (8, 20, 50), "scaling_repetitions": 2}
# HYBRID_OVERRIDES = {"max_candidates_per_order": 2, "num_reads": 64}
# EXACT_LNS_OVERRIDES = {"iterations": 3, "local_time_limit_seconds": 10}
PROFILE_OVERRIDES: dict[str, object] = {}
HYBRID_OVERRIDES: dict[str, object] = {}
EXACT_LNS_OVERRIDES: dict[str, object] = {}
ENABLED_EXPERIMENTS = {
    "solver_comparison": True,
    "size_scaling": True,
    "synthetic_scaling": True,
    "candidate_dc_scope_sensitivity": True,
    "penalty_weight_sensitivity": True,
    "qubo_penalty_sensitivity": True,
    "candidate_count_sensitivity": True,
    "inventory_shock": True,
    "qubo_coefficient_noise": True,
    "qaoa_readout_noise": True,
    "pareto_pruning_ablation": True,
    "batch_strategy_ablation": True,
    "sampler_ablation": True,
    "synthetic_coordination_control": True,
}
ENABLE_GPU_BENCHMARK = False
ENABLE_IBM_HARDWARE = False  # True submits synthetic-only circuits to IBM Quantum
IBM_HARDWARE_PROFILE = "quick"  # "quick" = 6 jobs; "presentation" = 18
IBM_SHOTS = 512
IBM_BACKEND_NAME = None  # None selects the least-busy eligible backend
OUTPUT_ROOT = challenge_results_root(PROJECT_ROOT, producer="notebook")

print(
    {
        "project_root": str(PROJECT_ROOT),
        "bundle_dir": str(BUNDLE_DIR),
        "profile": PROFILE,
        "force_rerun": FORCE_RERUN,
        "enabled_experiments": sum(ENABLED_EXPERIMENTS.values()),
        "notebook_results_root": str(OUTPUT_ROOT),
        "ibm_hardware_enabled": ENABLE_IBM_HARDWARE,
        "ibm_hardware_profile": IBM_HARDWARE_PROFILE,
        "requested_ibm_backend": IBM_BACKEND_NAME or "least_busy",
    }
)


## 1. Runtime-input readability gate

**Purpose.** Verify that the five required runtime tables are present, readable, and
non-empty before any modeling work begins. The reference PDF, equations document,
and workbook are not solver inputs; two recommendation outputs are optional audits.

**Why it matters.** A solver result is not reproducible when an upload is missing, a
numbered duplicate is selected accidentally, or a metadata sidecar is read as data.
Use `scripts/prepare_challenge_bundle.py` once if downloaded filenames need cleanup.


In [ ]:
file_audit = audit_poc_bundle(BUNDLE_DIR)
assert file_audit["readable"].all()
display(
    file_audit[
        ["role", "filename", "rows", "columns", "readable"]
    ]
)


## 2. Build and audit the real POC model

**Purpose.** Convert planning units to integer cases, identify focus loads, create
eligible DC/date options, protect five days of inventory, and apply documented dock
and penalty rules. Pareto pruning remains disabled here so it can be tested later.

**Why it matters.** This gate proves the optimization instance has the intended grain,
keys, candidate universe, and resource accounting before comparing algorithms.


In [ ]:
problem_unpruned = load_poc_problem(
    BUNDLE_DIR,
    config=POC_SETTINGS,
    strict_bundle_audit=False,
)
problem_pruned = prune_pareto_candidates(problem_unpruned)
summary = pd.DataFrame(
    [
        {
            "variant": "unpruned",
            "orders": len(problem_unpruned.orders),
            "assignment_groups": problem_unpruned.orders[
                "assignment_group"
            ].nunique(),
            "order_lines": len(problem_unpruned.order_lines),
            "candidate_rows": len(problem_unpruned.candidates),
        },
        {
            "variant": "pareto_pruned",
            "orders": len(problem_pruned.orders),
            "assignment_groups": problem_pruned.orders[
                "assignment_group"
            ].nunique(),
            "order_lines": len(problem_pruned.order_lines),
            "candidate_rows": len(problem_pruned.candidates),
        },
    ]
)
display(summary)

reference_available = all(
    (BUNDLE_DIR / name).is_file()
    for name in POC_REFERENCE_FILENAMES.values()
)
if reference_available:
    display(
        pd.Series(
            audit_poc_outputs(BUNDLE_DIR, problem_unpruned),
            name="reference audit",
        )
    )
else:
    print(
        "Optional recommendation outputs are absent; "
        "reconciliation is skipped."
    )


## 3. Verified experiment runner

**Purpose.** Run or resume each enabled study and write one easy-to-find table under
`results/challenge-study/notebook/<profile>/tables/`. Set `FORCE_RERUN = True` in the
global cell to bypass checkpoints.

**Why it matters.** Each manifest still verifies the complete configuration, problem,
source state, schema, columns, row count, and content hash, while stable profile paths
avoid opaque hash-directory proliferation. Infeasible rows are rejected before use.


In [ ]:
experiment_frames: dict[str, pd.DataFrame] = {}
BASE_PROFILE = experiment_profile(PROFILE)
if {"hybrid", "exact_lns"} & set(PROFILE_OVERRIDES):
    raise ValueError(
        "Use HYBRID_OVERRIDES or EXACT_LNS_OVERRIDES for nested solver settings"
    )
PROFILE_SETTINGS = replace(
    BASE_PROFILE,
    **PROFILE_OVERRIDES,
    hybrid=replace(BASE_PROFILE.hybrid, **HYBRID_OVERRIDES),
    exact_lns=replace(BASE_PROFILE.exact_lns, **EXACT_LNS_OVERRIDES),
)
PROFILE_CONFIGURATION = {
    "experiment_profile": asdict(PROFILE_SETTINGS),
    "poc": asdict(POC_SETTINGS),
}
SUITE_IDENTITY = checkpoint_identity(
    problem_unpruned,
    profile=PROFILE,
    experiment="challenge_suite",
    configuration=PROFILE_CONFIGURATION,
)
OUTPUT_DIR = checkpoint_run_directory(OUTPUT_ROOT, SUITE_IDENTITY)
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"artifact scope: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")


def feasible_mask(frame: pd.DataFrame) -> pd.Series:
    values = frame["feasible"]
    if values.dtype == bool:
        return values
    return values.astype(str).str.lower().isin({"true", "1"})


def run_or_load(name: str) -> pd.DataFrame | None:
    if not ENABLED_EXPERIMENTS.get(name, False):
        print(f"skipped by global switch: {name}")
        return None
    identity = checkpoint_identity(
        problem_unpruned,
        profile=PROFILE,
        experiment=name,
        configuration=PROFILE_CONFIGURATION,
    )
    path = TABLE_DIR / f"{name}.csv"
    frame = None
    if not FORCE_RERUN:
        try:
            frame = load_checkpoint(path, identity)
            print(f"loaded verified checkpoint: {path.relative_to(PROJECT_ROOT)}")
        except StaleCheckpointError as error:
            print(f"checkpoint unavailable or stale ({error}); recomputing {name}")
    if frame is None:
        frame = run_challenge_experiments(
            problem_unpruned,
            profile=PROFILE_SETTINGS,
            experiments=[name],
        )
        write_experiment_results(frame, path)
        write_checkpoint(frame, path, identity)
        print(f"wrote verified checkpoint: {path.relative_to(PROJECT_ROOT)}")

    invalid = frame.loc[~feasible_mask(frame)]
    if not invalid.empty:
        diagnostic_columns = [
            "experiment",
            "level",
            "validation_categories",
            "validation_violation_count",
            "error_type",
        ]
        columns = [column for column in diagnostic_columns if column in invalid]
        raise RuntimeError(
            "Infeasible experiment rows: "
            f"{invalid[columns].to_dict('records')}"
        )
    experiment_frames[name] = frame
    return frame


def show_result(
    frame: pd.DataFrame | None,
    columns: list[str],
    *,
    sort_by: str | list[str] | None = None,
    ascending: bool | list[bool] = True,
) -> None:
    if frame is None:
        display(Markdown("_Skipped by the global experiment switches._"))
        return
    available = [column for column in columns if column in frame.columns]
    view = frame[available]
    if sort_by is not None:
        view = view.sort_values(sort_by, ascending=ascending)
    display(view)


## 4. Common solver comparison

**Purpose.** Compare default routing, load-atomic greedy, polished greedy, adaptive
exact-MILP LNS, full exact MILP, and sampler-assisted LNS with one objective and one
independent validator.

**Why it matters.** This is the central fairness test and determines the production
hierarchy. It separates exact quantity-polish gains from assignment-search gains and
reports normalized capture alongside source-currency totals.


In [ ]:
solver_results = run_or_load("solver_comparison")
show_result(
    solver_results,
    [
        "method",
        "feasible",
        "objective_value",
        "requested_value",
        "objective_capture_rate",
        "case_fill_rate",
        "reassigned_orders",
        "penalty_cost",
        "shipping_cost",
        "runtime_seconds",
        "optimality_gap",
        "initial_polish_improvement",
        "search_improvement",
        "maximum_local_variables",
        "maximum_qubo_variables",
    ],
    sort_by="objective_value",
    ascending=False,
)


## 5. Optional MILP-backend comparison

**Purpose.** Solve the same exact compiled model with the license-free SciPy/HiGHS
backend and, when installed and licensed, Gurobi. The objective, bounds, integrality,
constraints, time limit, and independent validator are identical.

**Why it matters.** This isolates implementation speed from model quality. Gurobi is
never required or selected by default; unavailable commercial software is reported as
a transparent skip rather than causing the notebook to fail.


In [ ]:
backend_problem = select_shortage_subset(
    problem_unpruned, PROFILE_SETTINGS.base_orders
)
backend_rows: list[dict[str, object]] = []
for backend in ("scipy-highs", "gurobi"):
    if backend == "gurobi" and not available_milp_backends()["gurobi"]:
        backend_rows.append(
            {
                "milp_backend": backend,
                "status": "skipped: gurobipy is not installed",
                "feasible": None,
            }
        )
        continue
    try:
        solution = solve_classical(
            backend_problem,
            backend=backend,
            time_limit_seconds=60,
            mip_relative_gap=0.01,
            seed=PROFILE_SETTINGS.exact_lns.seed,
        )
    except ClassicalSolverError as error:
        backend_rows.append(
            {
                "milp_backend": backend,
                "status": f"skipped: {error}",
                "feasible": None,
            }
        )
        continue
    metrics = compute_metrics(backend_problem, solution)
    backend_rows.append(
        {
            "milp_backend": backend,
            "status": "completed",
            "feasible": metrics["feasible"],
            "objective_value": metrics["objective_value"],
            "objective_capture_rate": metrics["objective_capture_rate"],
            "case_fill_rate": metrics["case_fill_rate"],
            "runtime_seconds": metrics["runtime_seconds"],
            "optimality_gap": metrics["optimality_gap"],
        }
    )

backend_comparison = pd.DataFrame(backend_rows)
completed_backends = backend_comparison.loc[
    backend_comparison["status"].eq("completed")
]
assert completed_backends["feasible"].all()
if len(completed_backends) == 2:
    objective_spread = (
        completed_backends["objective_value"].max()
        - completed_backends["objective_value"].min()
    )
    assert abs(objective_spread) <= 1e-6 * max(
        1.0, completed_backends["objective_value"].abs().max()
    )
backend_comparison.to_csv(
    TABLE_DIR / "milp_backend_comparison.csv", index=False
)
display(backend_comparison)


## 6. Real assignment-group size scaling

**Purpose.** Scale greedy, polished greedy, bounded exact LNS, tractable hybrid LNS,
and small-instance exact MILP over nested real assignment-group subsets.

**Why it matters.** A load can contain several orders, so assignment groups—not raw
rows—are the true decision count. Repetitions, stage runtimes, variable growth, and
normalized quality show where each method remains practical.


In [ ]:
scaling_results = run_or_load("size_scaling")
show_result(
    scaling_results,
    [
        "method",
        "actual_assignment_groups",
        "repetition",
        "order_count",
        "order_line_count",
        "candidate_count",
        "maximum_local_variables",
        "maximum_qubo_variables",
        "objective_value",
        "objective_capture_rate",
        "case_fill_rate",
        "runtime_seconds",
        "feasible",
    ],
    sort_by=["actual_assignment_groups", "method", "repetition"],
)


## 7. Controlled synthetic scaling

**Purpose.** Generate independent coupled instances at each size and compare greedy,
exact LNS, tractable hybrid LNS, and small-instance exact MILP.

**Why it matters.** Nested real subsets confound size with composition. This control
is valid for complexity and variable-growth claims, but never for real-business impact.


In [ ]:
synthetic_scaling_results = run_or_load("synthetic_scaling")
show_result(
    synthetic_scaling_results,
    [
        "method",
        "actual_assignment_groups",
        "repetition",
        "generator_seed",
        "candidate_count",
        "maximum_local_variables",
        "maximum_qubo_variables",
        "objective_capture_rate",
        "runtime_seconds",
        "feasible",
    ],
    sort_by=["actual_assignment_groups", "method", "repetition"],
)


## 8. Candidate-DC universe sensitivity

**Purpose.** Compare the legacy focus/default-DC candidate universe with every DC in
the shipping, inventory, and dock-capacity intersection.

**Why it matters.** Candidate generation can silently cap solution quality. Equal
results are useful negative evidence; different results quantify that restriction.


In [ ]:
candidate_scope_results = run_or_load(
    "candidate_dc_scope_sensitivity"
)
show_result(
    candidate_scope_results,
    [
        "candidate_dc_scope",
        "method",
        "candidate_count",
        "objective_capture_rate",
        "case_fill_rate",
        "runtime_seconds",
        "feasible",
    ],
    sort_by=["candidate_dc_scope", "method"],
)


## 9. Business penalty-weight sensitivity

**Purpose.** Scale unmet-demand penalties on a fixed set of loads with active penalty
exposure and compare routing, fill, penalty, and shipping outcomes.

**Why it matters.** This reveals whether recommendations are robust or driven by one
arbitrary business coefficient. Raw objectives across penalty scales are deliberately
not compared; the operational trade-offs are.


In [ ]:
business_penalty_results = run_or_load(
    "penalty_weight_sensitivity"
)
show_result(
    business_penalty_results,
    [
        "penalty_scale",
        "method",
        "case_fill_rate",
        "reassigned_orders",
        "penalty_cost",
        "shipping_cost",
        "runtime_seconds",
    ],
    sort_by=["penalty_scale", "method"],
)


## 10. QUBO penalty calibration

**Purpose.** Sweep one-hot and shared-resource QUBO penalty multipliers while measuring
raw validity, repair burden, validated improvement, and runtime.

**Why it matters.** These are algorithmic penalties—not business costs—and poor values
can dominate the proposal energy. Calibration prevents an arbitrary encoding choice.


In [ ]:
qubo_penalty_results = run_or_load("qubo_penalty_sensitivity")
show_result(
    qubo_penalty_results,
    [
        "one_hot_penalty_multiplier",
        "pair_penalty_multiplier",
        "raw_one_hot_rate",
        "hybrid_improvement",
        "accepted_moves",
        "recourse_solves",
        "runtime_seconds",
    ],
    sort_by=["one_hot_penalty_multiplier", "pair_penalty_multiplier"],
)


## 11. Candidate-count sensitivity

**Purpose.** Vary the retained DC/date alternatives per assignment group and measure
quality, QUBO width, and runtime.

**Why it matters.** Candidate caps are the main scalability lever. This identifies the
point where extra alternatives stop paying for their computational cost.


In [ ]:
candidate_results = run_or_load("candidate_count_sensitivity")
show_result(
    candidate_results,
    [
        "candidate_limit",
        "method",
        "candidate_count",
        "maximum_qubo_variables",
        "objective_value",
        "case_fill_rate",
        "runtime_seconds",
    ],
    sort_by=["candidate_limit", "method"],
)


## 12. Inventory-shock robustness

**Purpose.** Compare nominal routing with exact quantity recourse against policies
reoptimized after each inventory shock.

**Why it matters.** This separates fixed-policy robustness from wait-and-see recourse
without inventing an uncalibrated risk coefficient or double-counting shortage risk.


In [ ]:
shock_results = run_or_load("inventory_shock")
show_result(
    shock_results,
    [
        "inventory_shock",
        "method",
        "objective_value",
        "objective_capture_rate",
        "case_fill_rate",
        "unassigned_orders",
        "penalty_cost",
        "runtime_seconds",
    ],
    sort_by=["inventory_shock", "method"],
)


## 13. Seed and local QUBO coefficient-noise robustness

**Purpose.** Repeat sampler seeds while perturbing local QUBO coefficients, then apply
the same exact quantity recourse and global validation.

**Why it matters.** It measures proposal stability under an analog/control proxy while
keeping the limit explicit: this is not a complete physical-device noise model.


In [ ]:
noise_results = run_or_load("qubo_coefficient_noise")
show_result(
    noise_results,
    [
        "seed",
        "coefficient_noise_relative_sigma",
        "raw_one_hot_rate",
        "hybrid_improvement",
        "accepted_moves",
        "runtime_seconds",
    ],
    sort_by=["coefficient_noise_relative_sigma", "seed"],
)


## 14. Local QAOA readout-noise proxy

**Purpose.** Apply independent bit flips at measurement to ideal Dicke/XY samples,
before deterministic one-hot repair, and track raw validity and final improvement.

**Why it matters.** It isolates readout sensitivity while clearly excluding gate error,
decoherence, and other effects that require simulator or hardware evidence.


In [ ]:
readout_noise_results = run_or_load("qaoa_readout_noise")
show_result(
    readout_noise_results,
    [
        "seed",
        "qaoa_readout_bitflip_probability",
        "raw_one_hot_rate",
        "hybrid_improvement",
        "accepted_moves",
        "runtime_seconds",
        "feasible",
    ],
    sort_by=["qaoa_readout_bitflip_probability", "seed"],
)


## 15. Heuristic Pareto-pruning ablation

**Purpose.** Compare unpruned candidates with an explicitly heuristic isolated-score
Pareto reduction and revalidate both solutions.

**Why it matters.** Options consume different shared resources, so this pruning is not
globally lossless. The ablation quantifies observed width/speed benefits and quality risk.


In [ ]:
pruning_results = run_or_load("pareto_pruning_ablation")
show_result(
    pruning_results,
    [
        "level",
        "candidate_count",
        "maximum_qubo_variables",
        "hybrid_improvement",
        "runtime_seconds",
        "feasible",
    ],
)


## 16. Random versus conflict-based batches

**Purpose.** Compare random neighborhoods with batches that group loads competing for
the same inventory or capacity under identical QUBO and runtime limits.

**Why it matters.** A local move is useful only when its decisions interact; this tests
whether problem-aware decomposition improves proposal quality.


In [ ]:
batch_results = run_or_load("batch_strategy_ablation")
show_result(
    batch_results,
    [
        "level",
        "hybrid_improvement",
        "accepted_moves",
        "maximum_qubo_variables",
        "recourse_solves",
        "runtime_seconds",
    ],
)


## 17. Sampler and quantum-simulation ablation

**Purpose.** On one coupled synthetic control, compare random and annealing samplers,
exact feasible enumeration, and local gate-model Dicke/XY-QAOA with identical recourse.

**Why it matters.** Holding the downstream solver fixed isolates proposal quality and
checks that the constraint-preserving encoding behaves as intended.


In [ ]:
sampler_results = run_or_load("sampler_ablation")
show_result(
    sampler_results,
    [
        "level",
        "raw_one_hot_rate",
        "initial_polish_improvement",
        "hybrid_improvement",
        "accepted_moves",
        "quantum_simulator_calls",
        "runtime_seconds",
        "feasible",
    ],
)


## 18. Synthetic coordination control

**Purpose.** Solve an independently generated coupled case designed to expose greedy
myopia and compare against an exact reference.

**Why it matters.** It verifies that coordinated search can improve an incumbent even
when the real subset does not show a gain. It cannot support real-data or quantum-advantage claims.


In [ ]:
synthetic_results = run_or_load(
    "synthetic_coordination_control"
)
show_result(
    synthetic_results,
    [
        "method",
        "objective_value",
        "case_fill_rate",
        "runtime_seconds",
        "optimality_gap",
        "raw_initial_objective",
        "initial_polish_improvement",
        "hybrid_improvement",
        "total_hybrid_improvement",
        "feasible",
    ],
    sort_by="objective_value",
    ascending=False,
)


## 19. Persist aggregate evidence and graphics

**Purpose.** Combine all enabled, manifest-verified experiments into one aggregate
table and create stable PNG evidence plots in the current profile directory.

**Why it matters.** A single discoverable table supports review and later report
generation while preserving per-experiment checkpoints and aggregate-only privacy.


In [ ]:
prepared_frames = [
    frame.dropna(axis=1, how="all")
    for frame in experiment_frames.values()
]
if prepared_frames:
    results = pd.concat(prepared_frames, ignore_index=True, sort=False)
    aggregate_identity = checkpoint_identity(
        problem_unpruned,
        profile=PROFILE,
        experiment="aggregate_results",
        configuration=PROFILE_CONFIGURATION,
    )
    aggregate_path = write_experiment_results(
        results, OUTPUT_DIR / "aggregate_results.csv"
    )
    write_checkpoint(results, aggregate_path, aggregate_identity)
    figure_paths = plot_challenge_results(results, FIGURE_DIR)
    print(
        f"wrote {len(results)} aggregate rows to "
        f"{aggregate_path.relative_to(PROJECT_ROOT)}"
    )
    for name, path in figure_paths.items():
        display(Markdown(f"### {name.replace('_', ' ').title()}"))
        display(Image(filename=str(path)))
    print(
        f"saved {len(figure_paths)} figures in "
        f"{FIGURE_DIR.relative_to(PROJECT_ROOT)}"
    )
else:
    print("No experiment families were enabled; no aggregate was written.")


## 20. Optional GPU crossover benchmark

**Purpose.** Measure synthetic batched QUBO energy scoring on CPU and, when available,
GPU; enable it with `ENABLE_GPU_BENCHMARK = True` in the global cell.

**Why it matters.** Current local QUBOs are small and exact recourse is CPU-based, so
GPU launch/transfer overhead may dominate. This locates a scoring crossover without
claiming end-to-end solver acceleration.


In [ ]:
capabilities = hardware_capabilities()
display(pd.Series(capabilities, name="hardware capability"))
if ENABLE_GPU_BENCHMARK:
    hardware_results = benchmark_qubo_batch_scoring(include_gpu=True)
    hardware_path = OUTPUT_DIR / "hardware_qubo_scoring.csv"
    hardware_results.to_csv(hardware_path, index=False)
    hardware_figure = plot_hardware_benchmark(
        hardware_results,
        FIGURE_DIR / "hardware_qubo_scoring.png",
    )
    display(hardware_results)
    display(Image(filename=str(hardware_figure)))
else:
    print("GPU benchmark skipped by the global switch.")


## 21. IBM backend discovery and hardware stress test

**Purpose.** Compare exact, local simulator, and IBM Dicke/XY-QAOA on the same
independently generated coupled control. The full matrix is two depths by three
mitigation strategies: `quick` submits six QPU jobs; `presentation` repeats them three
times for eighteen. Enable it at the top with `ENABLE_IBM_HARDWARE = True`.

**Why it matters.** It separates raw feasible-QUBO hit quality from exact recourse and
final validated gain, while recording compilation, queue, execution, quantum-use, and
decode timings. No Nestlé values or identifiers are sent. This is hardware-quality
evidence—not a quantum-advantage claim—and requires an already configured IBM account.


In [ ]:
if ENABLE_IBM_HARDWARE:
    logical_qubits = ibm_hardware_study_logical_qubits()
    backend_snapshot = discover_ibm_backends(min_num_qubits=logical_qubits)
    selected_backend = IBM_BACKEND_NAME or str(
        backend_snapshot.loc[
            backend_snapshot["selected_least_busy"], "backend"
        ].iloc[0]
    )
    if selected_backend not in set(backend_snapshot["backend"].astype(str)):
        raise RuntimeError(
            f"Requested backend {selected_backend!r} is not eligible for this circuit"
        )
    backend_snapshot["selected_for_study"] = (
        backend_snapshot["backend"].astype(str).eq(selected_backend)
    )
    ibm_problem = make_ibm_hardware_study_problem()
    qpu_identity = checkpoint_identity(
        ibm_problem,
        profile=f"ibm-{IBM_HARDWARE_PROFILE}",
        experiment="ibm_hardware_stress",
        configuration={
            "backend": selected_backend,
            "shots": IBM_SHOTS,
            "hardware_profile": IBM_HARDWARE_PROFILE,
            "logical_qubits": logical_qubits,
            "matrix": "p=1,2 x baseline,dd,dd+measurement-twirling",
            "data_scope": "independently generated synthetic control",
        },
    )
    IBM_OUTPUT_DIR = checkpoint_run_directory(OUTPUT_ROOT, qpu_identity)
    IBM_TABLE_DIR = IBM_OUTPUT_DIR / "tables"
    IBM_FIGURE_DIR = IBM_OUTPUT_DIR / "figures"
    IBM_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    IBM_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    backend_path = IBM_OUTPUT_DIR / "ibm_backend_snapshot.csv"
    backend_snapshot.to_csv(backend_path, index=False)
    backend_figure = plot_ibm_backend_snapshot(
        backend_snapshot,
        IBM_FIGURE_DIR / "ibm_backend_queue.png",
    )
    display(backend_snapshot)
    display(Image(filename=str(backend_figure)))

    qpu_target = IBM_TABLE_DIR / "ibm_hardware_stress.csv"
    existing_qpu_results = None
    if not FORCE_RERUN:
        try:
            existing_qpu_results = load_checkpoint(qpu_target, qpu_identity)
            print(f"resuming from {len(existing_qpu_results)} verified rows")
        except StaleCheckpointError as error:
            print(f"no reusable IBM checkpoint ({error})")
    qpu_results = run_ibm_hardware_study(
        allow_remote=True,
        backend_name=selected_backend,
        shots=IBM_SHOTS,
        profile=IBM_HARDWARE_PROFILE,
        progress_callback=lambda frame: write_checkpoint(
            frame, qpu_target, qpu_identity
        ),
        existing_results=existing_qpu_results,
    )
    qpu_path, _ = write_checkpoint(qpu_results, qpu_target, qpu_identity)
    strategy_ranking = rank_ibm_hardware_strategies(qpu_results)
    ranking_path = write_experiment_results(
        strategy_ranking,
        IBM_TABLE_DIR / "ibm_strategy_ranking.csv",
    )
    show_result(
        qpu_results,
        [
            "level",
            "hardware_backend",
            "hardware_mitigation_strategy",
            "qaoa_layers",
            "feasible",
            "error_type",
            "error_message",
            "search_improvement",
            "raw_one_hot_rate",
            "hardware_qubo_optimal_hit_rate",
            "hardware_transpiled_depth",
            "hardware_two_qubit_gates",
            "hardware_queue_seconds",
            "hardware_execution_seconds",
            "hardware_quantum_seconds",
            "hardware_returned_samples",
            "hardware_feasible_shots",
            "runtime_seconds",
        ],
    )
    display(strategy_ranking)
    if bool(strategy_ranking["selected_best_observed"].any()):
        best_observed = strategy_ranking.loc[
            strategy_ranking["selected_best_observed"]
        ].iloc[0]
        display(
            Markdown(
                f"**Best observed successful IBM strategy:** {best_observed['variant']} "
                f"(median exact feasible-QUBO raw hit rate "
                f"{100 * best_observed['hardware_qubo_optimal_hit_rate']:.1f}%)."
            )
        )
        qpu_figure = plot_ibm_hardware_study(
            qpu_results,
            IBM_FIGURE_DIR / "ibm_hardware_stress.png",
        )
        display(Image(filename=str(qpu_figure)))
    else:
        print("No successful IBM row is available to rank or plot; failed rows remain saved.")
    print(f"wrote {qpu_path.relative_to(PROJECT_ROOT)}")
    print(f"wrote {ranking_path.relative_to(PROJECT_ROOT)}")
else:
    print("IBM hardware study skipped by the global switch.")


## 22. Interpretation guardrails and next evidence

A valid conclusion requires every returned solution to pass the independent
validator, hybrid never to fall below its greedy incumbent, and real POC
evidence to remain separate from synthetic controls. Compare business-penalty
settings through fill, penalty, shipping, and reassignment trade-offs rather
than raw objectives across different scales.

Do not add a generic risk coefficient until forecast scenarios and
probabilities can calibrate it. If those become available, the next defensible
extension is a scenario-based expected-shortfall or CVaR model compared with
the inventory-shock frontier here. Do not generate the final report or
presentation until the full profile and any approved hardware runs complete.
